# SEC EDGAR Filing Collector

Pulls company filings from the **SEC EDGAR** free API.
Covers 1993-present for all publicly traded US companies.

## Filing types collected
| Type | Description | Signal |
|------|-------------|--------|
| 8-K | Material events (earnings, M&A, management changes) | High |
| 10-Q | Quarterly earnings report | High |
| 10-K | Annual report | Medium |
| 13-F | Institutional holdings changes | Medium |
| S-1 | IPO filing | Medium |

## Output files
| File | Contents |
|------|----------|
| `edgar_data/edgar_{TICKER}.csv` | All filings metadata per ticker |
| `edgar_data/edgar_index.csv` | Coverage summary |

---
## 0. Install

In [1]:
# !pip install requests pandas python-dotenv

---
## 1. Configuration

In [2]:
import os, io, time, base64, requests
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
load_dotenv()

GITHUB_REPO   = 'annhmartin/dataviz-historical-stocks-AnnetteMartin'
GITHUB_TOKEN  = os.environ.get('GITHUB_TOKEN', None)
EDGAR_PREFIX  = 'edgar_data'

# SEC requires a user-agent header — put your email here
YOUR_EMAIL    = 'your.email@example.com'   # CHANGE THIS
SEC_HEADERS   = {
    'User-Agent'    : f'TechPulse/1.0 ({YOUR_EMAIL})',
    'Accept-Encoding': 'gzip, deflate',
    'Host'          : 'data.sec.gov',
}

FILING_TYPES = ['8-K', '10-Q', '10-K', '13-F', 'S-1']
DELAY        = 0.1   # SEC rate limit: 10 requests/second max

# Ticker to CIK mapping (SEC uses CIK, not ticker symbols)
# We fetch this dynamically from SEC's company search API
TICKERS = [
    'AAPL','MSFT','GOOGL','META','AMZN','NVDA','TSM','INTC','AMD','QCOM',
    'TSLA','COIN','PYPL','NFLX','CRM','CRWD','PANW','PLTR','DDOG','SNOW',
    'MDB','NOW','OKTA','NVO','INCY','KGC','PM','WPM','SPOT','PINS',
]

print('Configuration loaded')
print(f'  Repo         : {GITHUB_REPO}')
print(f'  Token        : {"set" if GITHUB_TOKEN else "NOT SET"}')
print(f'  Filing types : {FILING_TYPES}')
print(f'  Tickers      : {len(TICKERS)}')
print(f'  User-Agent   : {SEC_HEADERS["User-Agent"]}')

Configuration loaded
  Repo         : annhmartin/dataviz-historical-stocks-AnnetteMartin
  Token        : set
  Filing types : ['8-K', '10-Q', '10-K', '13-F', 'S-1']
  Tickers      : 30
  User-Agent   : TechPulse/1.0 (your.email@example.com)


---
## 2. GitHub helpers

In [3]:
GITHUB_API = 'https://api.github.com'

def _gh_headers(token):
    return {'Authorization': f'Bearer {token}',
            'Accept': 'application/vnd.github+json',
            'X-GitHub-Api-Version': '2022-11-28'}

def push_csv(df, path, token, msg=None):
    if msg is None: msg = f'Update {path} - {len(df):,} rows'
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    encoded = base64.b64encode(buf.getvalue().encode()).decode()
    url = f'{GITHUB_API}/repos/{GITHUB_REPO}/contents/{path}'
    headers = _gh_headers(token)
    check = requests.get(url, headers=headers, timeout=15)
    sha = check.json().get('sha') if check.status_code == 200 else None
    payload = {'message': msg, 'content': encoded}
    if sha: payload['sha'] = sha
    for attempt in range(3):
        resp = requests.put(url, headers=headers, json=payload, timeout=120)
        if resp.status_code in (200, 201):
            print(f'  Saved {path} ({len(df):,} rows)')
            return True
        if resp.status_code == 409 and attempt < 2:
            time.sleep(3)
            check = requests.get(url, headers=headers, timeout=15)
            sha = check.json().get('sha') if check.status_code == 200 else None
            if sha: payload['sha'] = sha
        else:
            print(f'  FAILED {path}: {resp.status_code}')
            return False

def load_csv(path, token=None):
    url = f'https://raw.githubusercontent.com/{GITHUB_REPO}/main/{path}'
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.status_code == 404: raise FileNotFoundError(path)
    resp.raise_for_status()
    content = resp.text.strip()
    if not content: return pd.DataFrame()
    return pd.read_csv(io.StringIO(content), low_memory=False)

print('GitHub helpers loaded')

GitHub helpers loaded


---
## 3. SEC EDGAR API helpers

In [4]:
SEC_BASE = 'https://data.sec.gov'

def get_cik(ticker):
    """
    Look up the SEC CIK number for a ticker symbol.
    CIK is zero-padded to 10 digits.
    """
    url  = 'https://efts.sec.gov/LATEST/search-index?q=%22{}%22&dateRange=custom&startdt=2000-01-01&enddt=2000-01-02&forms=10-K'.format(ticker)
    # Use the company tickers JSON instead
    url  = 'https://www.sec.gov/files/company_tickers.json'
    resp = requests.get(url, headers={'User-Agent': SEC_HEADERS['User-Agent']}, timeout=15)
    if resp.status_code != 200:
        return None
    data = resp.json()
    for entry in data.values():
        if entry.get('ticker', '').upper() == ticker.upper():
            cik = str(entry['cik_str']).zfill(10)
            return cik
    return None

def get_filings(cik, filing_types):
    """
    Fetch all filing metadata for a company from EDGAR.
    Returns a list of filing dicts with date, type, and accession number.
    """
    url  = f'{SEC_BASE}/submissions/CIK{cik}.json'
    resp = requests.get(url, headers=SEC_HEADERS, timeout=30)
    if resp.status_code != 200:
        return []
    data = resp.json()
    filings = data.get('filings', {}).get('recent', {})
    if not filings:
        return []

    forms       = filings.get('form', [])
    dates       = filings.get('filingDate', [])
    accessions  = filings.get('accessionNumber', [])
    descriptions= filings.get('primaryDocument', [])
    sizes       = filings.get('size', [])

    results = []
    for i, form in enumerate(forms):
        if form in filing_types:
            results.append({
                'form'       : form,
                'date'       : dates[i] if i < len(dates) else None,
                'accession'  : accessions[i] if i < len(accessions) else None,
                'document'   : descriptions[i] if i < len(descriptions) else None,
                'size_bytes' : sizes[i] if i < len(sizes) else None,
            })
    return results

def get_older_filings(cik, filing_types):
    """
    EDGAR only returns recent filings in the main JSON.
    For older filings, query the full-text search API.
    """
    url = f'{SEC_BASE}/submissions/CIK{cik}.json'
    resp = requests.get(url, headers=SEC_HEADERS, timeout=30)
    if resp.status_code != 200: return []
    data = resp.json()
    # Check for additional filing pages
    older = data.get('filings', {}).get('files', [])
    all_results = []
    for file_info in older:
        file_url = f"{SEC_BASE}/submissions/{file_info['name']}"
        r = requests.get(file_url, headers=SEC_HEADERS, timeout=30)
        if r.status_code != 200: continue
        d = r.json()
        forms      = d.get('form', [])
        dates      = d.get('filingDate', [])
        accessions = d.get('accessionNumber', [])
        docs       = d.get('primaryDocument', [])
        sizes      = d.get('size', [])
        for i, form in enumerate(forms):
            if form in filing_types:
                all_results.append({
                    'form'      : form,
                    'date'      : dates[i] if i < len(dates) else None,
                    'accession' : accessions[i] if i < len(accessions) else None,
                    'document'  : docs[i] if i < len(docs) else None,
                    'size_bytes': sizes[i] if i < len(sizes) else None,
                })
        time.sleep(DELAY)
    return all_results

# Build CIK lookup table
print('Building CIK lookup table ...')
url  = 'https://www.sec.gov/files/company_tickers.json'
resp = requests.get(url, headers={'User-Agent': SEC_HEADERS['User-Agent']}, timeout=15)
cik_data = resp.json()
ticker_to_cik = {}
for entry in cik_data.values():
    t = entry.get('ticker', '').upper()
    if t in [tk.upper() for tk in TICKERS]:
        ticker_to_cik[t] = str(entry['cik_str']).zfill(10)

print(f'Found CIKs for {len(ticker_to_cik)}/{len(TICKERS)} tickers')
for t, cik in sorted(ticker_to_cik.items()):
    print(f'  {t:8s} -> CIK {cik}')

Building CIK lookup table ...
Found CIKs for 30/30 tickers
  AAPL     -> CIK 0000320193
  AMD      -> CIK 0000002488
  AMZN     -> CIK 0001018724
  COIN     -> CIK 0001679788
  CRM      -> CIK 0001108524
  CRWD     -> CIK 0001535527
  DDOG     -> CIK 0001561550
  GOOGL    -> CIK 0001652044
  INCY     -> CIK 0000879169
  INTC     -> CIK 0000050863
  KGC      -> CIK 0000701818
  MDB      -> CIK 0001441816
  META     -> CIK 0001326801
  MSFT     -> CIK 0000789019
  NFLX     -> CIK 0001065280
  NOW      -> CIK 0001373715
  NVDA     -> CIK 0001045810
  NVO      -> CIK 0000353278
  OKTA     -> CIK 0001660134
  PANW     -> CIK 0001327567
  PINS     -> CIK 0001506293
  PLTR     -> CIK 0001321655
  PM       -> CIK 0001413329
  PYPL     -> CIK 0001633917
  QCOM     -> CIK 0000804328
  SNOW     -> CIK 0001640147
  SPOT     -> CIK 0001639920
  TSLA     -> CIK 0001318605
  TSM      -> CIK 0001046179
  WPM      -> CIK 0001323404


---
## 4. Full Historical Fetch — All Filing Types

> Fetches all filing metadata for every ticker.
> Stores date, filing type, and accession number (not full document text).
> Full document text can be fetched later per-filing if needed.
> **Estimated time:** 15-30 minutes for all tickers (EDGAR is fast).

In [5]:
if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set.')
else:
    index_rows = []

    for ticker in TICKERS:
        cik = ticker_to_cik.get(ticker.upper())
        if not cik:
            print(f'  {ticker}: no CIK found — skipping')
            continue

        path = f'{EDGAR_PREFIX}/edgar_{ticker}.csv'

        # Check if already stored
        try:
            df_existing = load_csv(path, GITHUB_TOKEN)
            print(f'  {ticker}: already stored ({len(df_existing):,} filings) — updating')
        except FileNotFoundError:
            df_existing = pd.DataFrame()

        # Fetch recent filings
        filings = get_filings(cik, FILING_TYPES)
        time.sleep(DELAY)

        # Fetch older filings
        older = get_older_filings(cik, FILING_TYPES)
        all_filings = filings + older

        if not all_filings:
            print(f'  {ticker}: no filings found')
            continue

        df_new = pd.DataFrame(all_filings)
        df_new['ticker'] = ticker
        df_new['cik']    = cik
        df_new['edgar_url'] = df_new['accession'].apply(
            lambda a: f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{a.replace('-','')}/{a}-index.htm"
            if pd.notna(a) else None
        )

        # Merge with existing
        if not df_existing.empty:
            df_combined = (
                pd.concat([df_existing, df_new], ignore_index=True)
                .drop_duplicates(subset=['accession'])
                .sort_values('date', ascending=False)
                .reset_index(drop=True)
            )
        else:
            df_combined = df_new.sort_values('date', ascending=False).reset_index(drop=True)

        push_csv(df_combined, path, GITHUB_TOKEN,
                 f'EDGAR {ticker}: {len(df_combined):,} filings')

        index_rows.append({
            'ticker'        : ticker,
            'cik'           : cik,
            'filing_count'  : len(df_combined),
            'earliest_date' : df_combined['date'].min(),
            'latest_date'   : df_combined['date'].max(),
            'form_types'    : ', '.join(df_combined['form'].unique()),
        })
        time.sleep(DELAY)

    if index_rows:
        df_idx = pd.DataFrame(index_rows)
        push_csv(df_idx, f'{EDGAR_PREFIX}/edgar_index.csv', GITHUB_TOKEN,
                 f'EDGAR index: {df_idx["filing_count"].sum():,} total filings')
        print(f'\nDone!')
        print(df_idx[['ticker','filing_count','earliest_date','latest_date']].to_string(index=False))

  AAPL: already stored (359 filings) — updating
  Saved edgar_data/edgar_AAPL.csv (359 rows)
  MSFT: already stored (409 filings) — updating
  Saved edgar_data/edgar_MSFT.csv (409 rows)
  GOOGL: already stored (154 filings) — updating
  Saved edgar_data/edgar_GOOGL.csv (154 rows)
  META: already stored (187 filings) — updating
  Saved edgar_data/edgar_META.csv (187 rows)
  AMZN: already stored (379 filings) — updating
  Saved edgar_data/edgar_AMZN.csv (379 rows)
  NVDA: already stored (347 filings) — updating
  Saved edgar_data/edgar_NVDA.csv (347 rows)
  TSM: no filings found
  INTC: already stored (573 filings) — updating
  Saved edgar_data/edgar_INTC.csv (573 rows)
  AMD: already stored (546 filings) — updating
  Saved edgar_data/edgar_AMD.csv (546 rows)
  QCOM: already stored (377 filings) — updating
  Saved edgar_data/edgar_QCOM.csv (377 rows)
  TSLA: already stored (308 filings) — updating
  Saved edgar_data/edgar_TSLA.csv (308 rows)
  COIN: already stored (85 filings) — updating

---
## 5. Incremental Update — New Filings Only

In [6]:
if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set.')
else:
    print('Updating EDGAR filings ...')
    for ticker in TICKERS:
        cik = ticker_to_cik.get(ticker.upper())
        if not cik: continue
        path = f'{EDGAR_PREFIX}/edgar_{ticker}.csv'
        try:
            df_existing = load_csv(path, GITHUB_TOKEN)
            latest_date = df_existing['date'].max()
        except FileNotFoundError:
            df_existing = pd.DataFrame()
            latest_date = '1993-01-01'

        filings = get_filings(cik, FILING_TYPES)
        time.sleep(DELAY)
        if not filings: continue

        df_new = pd.DataFrame(filings)
        df_new['ticker'] = ticker
        df_new['cik']    = cik
        df_new = df_new[df_new['date'] > latest_date]

        if df_new.empty:
            print(f'  {ticker}: already current')
            continue

        df_combined = (
            pd.concat([df_existing, df_new], ignore_index=True)
            .drop_duplicates(subset=['accession'])
            .sort_values('date', ascending=False)
            .reset_index(drop=True)
        )
        push_csv(df_combined, path, GITHUB_TOKEN,
                 f'EDGAR {ticker} update: +{len(df_new)} new filings')
        print(f'  {ticker}: +{len(df_new)} new filings')

Updating EDGAR filings ...
  AAPL: already current
  MSFT: already current
  GOOGL: already current
  META: already current
  AMZN: already current
  NVDA: already current
  INTC: already current
  AMD: already current
  QCOM: already current
  TSLA: already current
  COIN: already current
  PYPL: already current
  NFLX: already current
  CRM: already current
  CRWD: already current
  PANW: already current
  PLTR: already current
  DDOG: already current
  SNOW: already current
  MDB: already current
  NOW: already current
  OKTA: already current
  INCY: already current
  PM: already current
  PINS: already current


---
## 6. Load & Verify Coverage

In [7]:
df_idx = load_csv(f'{EDGAR_PREFIX}/edgar_index.csv', GITHUB_TOKEN)
print(f'Total filings: {df_idx["filing_count"].sum():,}')
print(df_idx.to_string(index=False))

# Filing type breakdown
all_frames = []
for ticker in TICKERS[:5]:  # sample first 5
    try:
        df_t = load_csv(f'{EDGAR_PREFIX}/edgar_{ticker}.csv', GITHUB_TOKEN)
        all_frames.append(df_t)
    except FileNotFoundError:
        pass

if all_frames:
    df_sample = pd.concat(all_frames, ignore_index=True)
    print('\nFiling type breakdown (sample of 5 tickers):')
    print(df_sample.groupby(['ticker','form'])['accession'].count().to_string())

Total filings: 6,396
ticker     cik  filing_count earliest_date latest_date           form_types
  AAPL  320193           359    1994-01-26  2026-05-01      10-Q, 8-K, 10-K
  MSFT  789019           409    1994-02-14  2026-06-05      8-K, 10-Q, 10-K
 GOOGL 1652044           154    2015-10-22  2026-06-11      8-K, 10-Q, 10-K
  META 1326801           187    2012-02-01  2026-05-29 8-K, 10-Q, 10-K, S-1
  AMZN 1018724           379    1997-03-24  2026-06-12 8-K, 10-Q, 10-K, S-1
  NVDA 1045810           347    1998-03-06  2026-06-30 8-K, 10-Q, 10-K, S-1
  INTC   50863           573    1994-03-25  2026-05-15      8-K, 10-Q, 10-K
   AMD    2488           546    1994-01-27  2026-07-01      8-K, 10-Q, 10-K
  QCOM  804328           377    1996-05-13  2026-06-24      8-K, 10-Q, 10-K
  TSLA 1318605           308    2010-01-29  2026-04-23 10-Q, 8-K, 10-K, S-1
  COIN 1679788            85    2021-02-25  2026-06-18 8-K, 10-Q, 10-K, S-1
  PYPL 1633917           184    2015-07-01  2026-05-21      8-K, 10

---
## 7. Fetch Full Filing Text (Optional)

The collector above stores only filing metadata (date, type, accession number).
Run this section to fetch and store the actual text of specific filing types
for sentiment analysis.

**Warning:** Full 10-K/10-Q documents can be very large. Only run for 8-K filings
unless you have a specific need for the longer documents.

In [8]:
def fetch_filing_text(cik, accession, primary_doc, max_chars=5000):
    """
    Fetch the text of a specific SEC filing using the primary document name.
    Pass primary_doc from the 'document' column in your edgar_{TICKER}.csv.
    """
    import re
    cik_int   = int(cik)
    acc_clean = accession.replace('-', '')
    doc_url   = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_clean}/{primary_doc}"

    resp = requests.get(doc_url, headers=SEC_HEADERS, timeout=30)
    time.sleep(DELAY)
    if resp.status_code != 200:
        return None

    text = re.sub(r'<[^>]+>', ' ', resp.text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text[:max_chars] if len(text) > 200 else None

FETCH_TEXT_FOR = 'AAPL'
FETCH_FORM     = '8-K'

try:
    df_t     = load_csv(f'{EDGAR_PREFIX}/edgar_{FETCH_TEXT_FOR}.csv', GITHUB_TOKEN)
    eight_ks = df_t[df_t['form'] == FETCH_FORM].dropna(subset=['document']).head(3)
    cik      = ticker_to_cik.get(FETCH_TEXT_FOR)
    for _, row in eight_ks.iterrows():
        text = fetch_filing_text(cik, row['accession'], row['document'])
        print(f"\n{row['date']} {row['form']}:")
        print(text[:300] if text else 'Could not fetch')
        time.sleep(DELAY)
except FileNotFoundError:
    print(f'Run Section 4 first to collect {FETCH_TEXT_FOR} filings')


2026-04-30 8-K:
Could not fetch

2026-04-20 8-K:
Could not fetch

2026-02-24 8-K:
Could not fetch


In [9]:
# Diagnostic — run this to see exactly what's happening
import requests, re

FETCH_TEXT_FOR = 'AAPL'
FETCH_FORM     = '8-K'

df_t     = load_csv(f'{EDGAR_PREFIX}/edgar_{FETCH_TEXT_FOR}.csv', GITHUB_TOKEN)
eight_ks = df_t[df_t['form'] == FETCH_FORM].dropna(subset=['document']).head(3)
cik      = ticker_to_cik.get(FETCH_TEXT_FOR)

print(f"CIK: {cik}")
print(f"\nFirst 3 rows from CSV:")
print(eight_ks[['date','form','accession','document']].to_string())

# Test the URL for the first one
row = eight_ks.iloc[0]
acc_clean = row['accession'].replace('-', '')
doc_url   = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_clean}/{row['document']}"
print(f"\nBuilt URL: {doc_url}")

resp = requests.get(doc_url, headers=SEC_HEADERS, timeout=30)
print(f"Status: {resp.status_code}")
print(f"Content-Type: {resp.headers.get('Content-Type', 'unknown')}")
print(f"Content length: {len(resp.content)}")
print(f"First 200 chars: {resp.text[:200]}")


CIK: 0000320193

First 3 rows from CSV:
         date form             accession           document
1  2026-04-30  8-K  0000320193-26-000011  aapl-20260430.htm
2  2026-04-20  8-K  0001140361-26-015711  ef20071035_8k.htm
3  2026-02-24  8-K  0001140361-26-006577  ef20060722_8k.htm

Built URL: https://www.sec.gov/Archives/edgar/data/320193/000032019326000011/aapl-20260430.htm
Status: 404
Content-Type: application/xml
Content length: 334
First 200 chars: <?xml version="1.0" encoding="UTF-8"?>
<Error><Code>NoSuchKey</Code><Message>The specified key does not exist.</Message><Key>Archives/edgar/data/320193/000032019326000011/aapl-20260430.htm</Key><Reque


In [10]:
# Diagnostic step 2 — check what's actually in the filing folder
row = eight_ks.iloc[0]
acc_clean = row['accession'].replace('-', '')
cik_int   = int(cik)

# The index page lists all documents in the filing
index_url = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_clean}/{row['accession']}-index.htm"
print(f"Index URL: {index_url}")
resp = requests.get(index_url, headers=SEC_HEADERS, timeout=15)
print(f"Status: {resp.status_code}")
print(resp.text[:1000])

Index URL: https://www.sec.gov/Archives/edgar/data/320193/000032019326000011/0000320193-26-000011-index.htm
Status: 404
<?xml version="1.0" encoding="UTF-8"?>
<Error><Code>NoSuchKey</Code><Message>The specified key does not exist.</Message><Key>Archives/edgar/data/320193/000032019326000011/0000320193-26-000011-index.htm</Key><RequestId>KN1JH5CVCN1RAEKE</RequestId><HostId>PGfkzSRXrvDvgauOQg88QRorMQiyNyO07zK/flrwMxiRVoObThd8an481UITLJA2YsI5SqXR2dU=</HostId></Error>


In [11]:
index_url = f"https://data.sec.gov/submissions/CIK{str(cik_int).zfill(10)}.json"
# Can't reach data.sec.gov — use the full-text search instead
search_url = f"https://efts.sec.gov/LATEST/search-index?q=%22{row['accession']}%22&forms=8-K"
print(f"Search URL: {search_url}")
resp = requests.get(search_url, headers={'User-Agent': SEC_HEADERS['User-Agent']}, timeout=15)
print(f"Status: {resp.status_code}")
print(resp.text[:500])

Search URL: https://efts.sec.gov/LATEST/search-index?q=%220000320193-26-000011%22&forms=8-K
Status: 200
{"took":580,"timed_out":false,"_shards":{"total":50,"successful":50,"skipped":0,"failed":0},"hits":{"total":{"value":2,"relation":"eq"},"max_score":0,"hits":[{"_index":"edgar_file","_id":"0000320193-26-000011:aapl-20260430.htm","_score":0,"_source":{"ciks":["0000320193"],"period_ending":"2026-04-30","file_num":["001-36743"],"display_names":["Apple Inc.  (AAPL)  (CIK 0000320193)"],"xsl":null,"sequence":1,"root_forms":["8-K"],"file_date":"2026-04-30","biz_states":["CA"],"sics":["3571"],"form":"8-K


In [12]:
# Test with the filing agent's CIK instead of Apple's CIK
row = eight_ks.iloc[1]  # second filing — filed by agent 0001140361
acc_clean   = row['accession'].replace('-', '')
agent_cik   = row['accession'].split('-')[0].lstrip('0')  # extract CIK from accession
print(f"Accession    : {row['accession']}")
print(f"Agent CIK    : {agent_cik}")
print(f"Company CIK  : {cik_int}")

# Try with company CIK
url1 = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_clean}/{row['document']}"
# Try with agent CIK  
url2 = f"https://www.sec.gov/Archives/edgar/data/{agent_cik}/{acc_clean}/{row['document']}"

for label, url in [("Company CIK", url1), ("Agent CIK", url2)]:
    r = requests.get(url, headers=SEC_HEADERS, timeout=15)
    print(f"\n{label}: {url}")
    print(f"Status: {r.status_code}")
    if r.status_code == 200:
        print(f"Content length: {len(r.content)}")
        print(f"First 100 chars: {r.text[:100]}")

Accession    : 0001140361-26-015711
Agent CIK    : 1140361
Company CIK  : 320193

Company CIK: https://www.sec.gov/Archives/edgar/data/320193/000114036126015711/ef20071035_8k.htm
Status: 404

Agent CIK: https://www.sec.gov/Archives/edgar/data/1140361/000114036126015711/ef20071035_8k.htm
Status: 404


In [13]:
# Use EDGAR full-text search to get the real document URL
import json

for _, row in eight_ks.iterrows():
    search_url = f"https://efts.sec.gov/LATEST/search-index?q=%22{row['accession']}%22&forms=8-K"
    resp = requests.get(search_url, 
                        headers={'User-Agent': SEC_HEADERS['User-Agent']}, 
                        timeout=15)
    if resp.status_code != 200:
        print(f"Search failed: {resp.status_code}")
        continue
    
    data = resp.json()
    hits = data.get('hits', {}).get('hits', [])
    
    for hit in hits:
        src = hit.get('_source', {})
        hit_id = hit.get('_id', '')
        # _id format is "accession:filename"
        print(f"\nDate: {row['date']}")
        print(f"ID: {hit_id}")
        print(f"Period: {src.get('period_ending')}")
        print(f"Form: {src.get('form')}")
        
        # Build URL from _id
        if ':' in hit_id:
            acc_part, fname = hit_id.split(':', 1)
            acc_clean = acc_part.replace('-', '')
            test_url = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_clean}/{fname}"
            print(f"Test URL: {test_url}")
            r = requests.get(test_url, headers=SEC_HEADERS, timeout=15)
            print(f"Status: {r.status_code}")
            if r.status_code == 200:
                print(f"SUCCESS - content length: {len(r.content)}")


Date: 2026-04-30
ID: 0000320193-26-000011:aapl-20260430.htm
Period: 2026-04-30
Form: 8-K
Test URL: https://www.sec.gov/Archives/edgar/data/320193/000032019326000011/aapl-20260430.htm
Status: 404

Date: 2026-04-30
ID: 0000320193-26-000011:a8-kex991q2202603282026.htm
Period: 2026-04-30
Form: 8-K
Test URL: https://www.sec.gov/Archives/edgar/data/320193/000032019326000011/a8-kex991q2202603282026.htm
Status: 404

Date: 2026-04-20
ID: 0001140361-26-015711:ef20071035_8k.htm
Period: 2026-04-17
Form: 8-K
Test URL: https://www.sec.gov/Archives/edgar/data/320193/000114036126015711/ef20071035_8k.htm
Status: 404
Search failed: 500


In [14]:
# Test if SEC.gov is reachable at all for document fetches
test_urls = [
    "https://www.sec.gov/robots.txt",
    "https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=AAPL&type=8-K&dateb=&owner=include&count=5&output=atom",
    "https://efts.sec.gov/LATEST/search-index?q=%22apple%22&forms=8-K&dateRange=custom&startdt=2024-01-01&enddt=2024-01-31",
]

for url in test_urls:
    r = requests.get(url, headers={'User-Agent': SEC_HEADERS['User-Agent']}, timeout=15)
    print(f"Status: {r.status_code} | {url[:80]}")
    if r.status_code == 200:
        print(f"  Content: {r.text[:100]}")

Status: 200 | https://www.sec.gov/robots.txt
  Content: #
# robots.txt
#
# This file is to prevent the crawling and indexing of certain parts
# of your site
Status: 200 | https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=AAPL&type=8-K&dat
  Content: <?xml version="1.0" encoding="ISO-8859-1" ?>
  <feed xmlns="http://www.w3.org/2005/Atom">
    <autho
Status: 200 | https://efts.sec.gov/LATEST/search-index?q=%22apple%22&forms=8-K&dateRange=custo
  Content: {"took":1087,"timed_out":false,"_shards":{"total":50,"successful":50,"skipped":0,"failed":0},"hits":


In [15]:
# Try EDGAR viewer and inline viewer URLs which use different paths
row = eight_ks.iloc[0]
acc_clean = row['accession'].replace('-', '')
cik_int   = int(cik)

test_urls = [
    # Viewer format
    f"https://www.sec.gov/cgi-bin/viewer?action=view&cik={cik_int}&type=8-K&dateb=&owner=include&count=40",
    # Inline XBRL viewer
    f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc_clean}/",
    # Try without leading zeros on CIK
    f"https://www.sec.gov/Archives/edgar/data/{str(cik_int).lstrip('0')}/{acc_clean}/{row['document']}",
    # Full text search direct link format
    f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{row['accession']}/{row['document']}",
]

for url in test_urls:
    r = requests.get(url, headers=SEC_HEADERS, timeout=15)
    print(f"Status: {r.status_code} | {url[:100]}")
    if r.status_code == 200:
        print(f"  Content: {r.text[:200]}")

Status: 404 | https://www.sec.gov/cgi-bin/viewer?action=view&cik=320193&type=8-K&dateb=&owner=include&count=40
Status: 404 | https://www.sec.gov/Archives/edgar/data/320193/000032019326000011/
Status: 404 | https://www.sec.gov/Archives/edgar/data/320193/000032019326000011/aapl-20260430.htm
Status: 404 | https://www.sec.gov/Archives/edgar/data/320193/0000320193-26-000011/aapl-20260430.htm


In [16]:
# Check if Archives domain is specifically blocked
test_urls = [
    "https://www.sec.gov/Archives/edgar/full-index/2024/QTR1/company.idx",
    "https://efts.sec.gov/LATEST/search-index?q=%220000320193-26-000011%22",
]
for url in test_urls:
    r = requests.get(url, headers={'User-Agent': SEC_HEADERS['User-Agent']}, timeout=15)
    print(f"Status: {r.status_code} | {url[:80]}")
    # Check for egress block header
    deny = r.headers.get('x-deny-reason', '')
    if deny:
        print(f"  Blocked: {deny}")

Status: 200 | https://www.sec.gov/Archives/edgar/full-index/2024/QTR1/company.idx
Status: 200 | https://efts.sec.gov/LATEST/search-index?q=%220000320193-26-000011%22
